In [0]:
from pyspark.sql import functions as F

ruta = "/Volumes/aerolinea/fuentes/landing"

## Lectura de **vuelos**

In [0]:
vuelos_bronze = (
    spark.read.csv(
        ruta + "/vuelos_diarios.csv", header=True,
        schema="""vuelo_id STRING, fecha STRING, origen_id STRING, destino_id STRING,
                  aeronave_id STRING, estado STRING, duracion_min STRING""",
        mode="FAILFAST", enforceSchema=False
    )
    .withColumn("archivo_origen", F.lit("vuelos_diarios.csv"))
    .withColumn("fecha_ingesta", F.current_timestamp())
)

display(vuelos_bronze.limit(5))

vuelo_id,fecha,origen_id,destino_id,aeronave_id,estado,duracion_min,archivo_origen,fecha_ingesta
V0001,2024-05-21,MAD,JFK,A005,a_tiempo,288,vuelos_diarios.csv,2026-10-24T20:28:06.571Z
V0002,2024-05-05,MAD,LHR,A002,cancelado,492,vuelos_diarios.csv,2026-10-24T20:28:06.571Z
V0003,2024-05-02,MAD,JFK,A004,a_tiempo,577,vuelos_diarios.csv,2026-10-24T20:28:06.571Z
V0004,2024-05-20,MAD,LHR,A004,cancelado,618,vuelos_diarios.csv,2026-10-24T20:28:06.571Z
V0005,2024-05-14,BCN,BCN,A010,retrasado,66,vuelos_diarios.csv,2026-10-24T20:28:06.571Z


## aeronaves

In [0]:
aeronaves_bronze = (
    spark.read.csv(
        ruta + "/aeronaves.csv", header=True,
        schema="aeronave_id STRING, modelo STRING, fabricante STRING, anio_fabricacion STRING",
        mode="FAILFAST", enforceSchema=False
    )
    .withColumn("archivo_origen", F.lit("aeronaves.csv"))
    .withColumn("fecha_ingesta", F.current_timestamp())
)

## aeropuertos

In [0]:
aeropuertos_bronze = (
    spark.read.csv(
        ruta + "/aeropuertos.csv", header=True,
        schema="aeropuerto_id STRING, nombre STRING, ciudad STRING, pais STRING, lat STRING, lon STRING",
        mode="FAILFAST", enforceSchema=False
    )
    .withColumn("archivo_origen", F.lit("aeropuertos.csv"))
    .withColumn("fecha_ingesta", F.current_timestamp())
)

## Mantenimiento

In [0]:
mantenimientos_bronze = (
    spark.read.csv(
        ruta + "/mantenimientos_rds.csv", header=True,
        schema="""mantenimiento_id STRING, aeronave_id STRING, fecha STRING,
                  tipo STRING, costo_usd STRING, duracion_hr STRING""",
        mode="FAILFAST", enforceSchema=False
    )
    .withColumn("archivo_origen", F.lit("mantenimientos_rds.csv"))
    .withColumn("fecha_ingesta", F.current_timestamp())
)

## Revisión antes de escribir

In [0]:
print("Vuelos:", vuelos_bronze.count())
print("Aeronaves:", aeronaves_bronze.count())
print("Aeropuertos:", aeropuertos_bronze.count())
print("Mantenimientos:", mantenimientos_bronze.count())
vuelos_bronze.printSchema()

Vuelos: 100
Aeronaves: 10
Aeropuertos: 5
Mantenimientos: 100
root
 |-- vuelo_id: string (nullable = true)
 |-- fecha: string (nullable = true)
 |-- origen_id: string (nullable = true)
 |-- destino_id: string (nullable = true)
 |-- aeronave_id: string (nullable = true)
 |-- estado: string (nullable = true)
 |-- duracion_min: string (nullable = true)
 |-- archivo_origen: string (nullable = false)
 |-- fecha_ingesta: timestamp (nullable = false)



## Escritura Bronze

In [0]:
vuelos_bronze.write.format("delta").mode("overwrite").saveAsTable("aerolinea.bronze.vuelos")

In [0]:
aeronaves_bronze.write.format("delta").mode("overwrite").saveAsTable("aerolinea.bronze.aeronaves")

In [0]:
aeropuertos_bronze.write.format("delta").mode("overwrite").saveAsTable("aerolinea.bronze.aeropuertos")

In [0]:
mantenimientos_bronze.write.format("delta").mode("overwrite").saveAsTable("aerolinea.bronze.mantenimientos")

In [0]:
%sql
SELECT 'vuelos' AS fuente, COUNT(*) AS registros FROM aerolinea.bronze.vuelos
UNION ALL SELECT 'aeronaves', COUNT(*) FROM aerolinea.bronze.aeronaves
UNION ALL SELECT 'aeropuertos', COUNT(*) FROM aerolinea.bronze.aeropuertos
UNION ALL SELECT 'mantenimientos', COUNT(*) FROM aerolinea.bronze.mantenimientos

fuente,registros
vuelos,100
aeronaves,10
aeropuertos,5
mantenimientos,100
